In [1]:
import numpy as np 
import pandas as pd
from decimal import ROUND_HALF_UP, Decimal


In [2]:
train=pd.read_csv("input/jpx-tokyo-stock-exchange-prediction/train_files/stock_prices.csv", parse_dates=['Date'])
train = train[train['Date']>=pd.to_datetime('2021-01-01')]
stock_list=pd.read_csv("input/jpx-tokyo-stock-exchange-prediction/stock_list.csv")

print("The training data begins on {} and ends on {}.\n".format(train.Date.min(),train.Date.max()))
#display(train.describe().style.format('{:,.2f}'))
train.head(5)

The training data begins on 2021-01-04 00:00:00 and ends on 2021-12-03 00:00:00.



,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target
1880531,20210104_1301,2021-01-04,1301,2951.0,2951.0,2913.0,2950.0,9700,1.0,NaN,False,0.011502
1880532,20210104_1332,2021-01-04,1332,428.0,429.0,416.0,421.0,1780500,1.0,NaN,False,0.019093
1880533,20210104_1333,2021-01-04,1333,2229.0,2231.0,2179.0,2202.0,112400,1.0,NaN,False,0.015075
1880534,20210104_1375,2021-01-04,1375,1701.0,1701.0,1672.0,1674.0,67900,1.0,NaN,False,-0.003503
1880535,20210104_1376,2021-01-04,1376,1597.0,1597.0,1577.0,1588.0,4500,1.0,NaN,False,-0.012033


In [3]:
def adjust_price(price):
    """
    Args:
        price (pd.DataFrame)  : pd.DataFrame include stock_price
    Returns:
        price DataFrame (pd.DataFrame): stock_price with generated AdjustedClose
    """
    # transform Date column into datetime
    price.loc[: ,"Date"] = pd.to_datetime(price.loc[: ,"Date"], format="%Y-%m-%d")

    def generate_adjusted_close(df):
        """
        Args:
            df (pd.DataFrame)  : stock_price for a single SecuritiesCode
        Returns:
            df (pd.DataFrame): stock_price with AdjustedClose for a single SecuritiesCode
        """
        # sort data to generate CumulativeAdjustmentFactor
        df = df.sort_values("Date", ascending=False)
        # generate CumulativeAdjustmentFactor
        df.loc[:, "CumulativeAdjustmentFactor"] = df["AdjustmentFactor"].cumprod()
        # generate AdjustedClose
        df.loc[:, "AdjustedClose"] = (
            df["CumulativeAdjustmentFactor"] * df["Close"]
        ).map(lambda x: float(
            Decimal(str(x)).quantize(Decimal('0.1'), rounding=ROUND_HALF_UP)
        ))
        # reverse order
        df = df.sort_values("Date")
        # to fill AdjustedClose, replace 0 into np.nan
        df.loc[df["AdjustedClose"] == 0, "AdjustedClose"] = np.nan
        # forward fill AdjustedClose
        df.loc[:, "AdjustedClose"] = df.loc[:, "AdjustedClose"].ffill()
        return df
    
    # generate AdjustedClose
    price = price.sort_values(["SecuritiesCode", "Date"])
    price = price.groupby("SecuritiesCode").apply(generate_adjusted_close).reset_index(drop=True)
    return price
    
train['ExpectedDividend'] = train['ExpectedDividend'].notna().astype('int64')
#train = train.fillna(0)
prices=adjust_price(train)
prices.head(5)


C:\Users\mhwu9\AppData\Local\Temp\ipykernel_6912\1365220610.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  price = price.groupby("SecuritiesCode").apply(generate_adjusted_close).reset_index(drop=True)


,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target,CumulativeAdjustmentFactor,AdjustedClose
0,20210104_1301,2021-01-04,1301,2951.0,2951.0,2913.0,2950.0,9700,1.0,0,False,0.011502,1.0,2950.0
1,20210105_1301,2021-01-05,1301,2950.0,2959.0,2935.0,2956.0,11000,1.0,0,False,-0.000334,1.0,2956.0
2,20210106_1301,2021-01-06,1301,2958.0,2990.0,2955.0,2990.0,23000,1.0,0,False,0.003346,1.0,2990.0
3,20210107_1301,2021-01-07,1301,2990.0,2995.0,2975.0,2989.0,20400,1.0,0,False,0.043681,1.0,2989.0
4,20210108_1301,2021-01-08,1301,2988.0,3020.0,2971.0,2999.0,29500,1.0,0,False,-0.006390,1.0,2999.0


In [4]:
def create_features(df):
    df=df.copy()
    col='AdjustedClose'
    periods=[5,10,20,30,50]
    for period in periods:
        df.loc[:,"Return_{}Day".format(period)] = df.groupby("SecuritiesCode")[col].pct_change(period)
        df.loc[:,"MovingAvg_{}Day".format(period)] = df.groupby("SecuritiesCode")[col].rolling(window=period).mean().values
        df.loc[:,"ExpMovingAvg_{}Day".format(period)] = df.groupby("SecuritiesCode")[col].ewm(span=period,adjust=False).mean().values
        df.loc[:,"Volatility_{}Day".format(period)] = np.log(df[col]).groupby(df["SecuritiesCode"]).diff().rolling(period).std()
    return df

price_features=create_features(df=prices)
#price_features.drop(['RowId','SupervisionFlag','AdjustmentFactor','CumulativeAdjustmentFactor','Close'],axis=1,inplace=True)

C:\Users\mhwu9\AppData\Local\Temp\ipykernel_6912\2094419175.py:6: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df.loc[:,"Return_{}Day".format(period)] = df.groupby("SecuritiesCode")[col].pct_change(period)
C:\Users\mhwu9\AppData\Local\Temp\ipykernel_6912\2094419175.py:6: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df.loc[:,"Return_{}Day".format(period)] = df.groupby("SecuritiesCode")[col].pct_change(period)
C:\Users\mhwu9\AppData\Local\Temp\ipykernel_6912\2094419175.py:6: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be rem

In [5]:
price_features.head(10)

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,...,ExpMovingAvg_20Day,Volatility_20Day,Return_30Day,MovingAvg_30Day,ExpMovingAvg_30Day,Volatility_30Day,Return_50Day,MovingAvg_50Day,ExpMovingAvg_50Day,Volatility_50Day
0,20210104_1301,2021-01-04,1301,2951.0,2951.0,2913.0,2950.0,9700,1.0,0,...,2950.000000,NaN,NaN,NaN,2950.000000,NaN,NaN,NaN,2950.000000,NaN
1,20210105_1301,2021-01-05,1301,2950.0,2959.0,2935.0,2956.0,11000,1.0,0,...,2950.571429,NaN,NaN,NaN,2950.387097,NaN,NaN,NaN,2950.235294,NaN
2,20210106_1301,2021-01-06,1301,2958.0,2990.0,2955.0,2990.0,23000,1.0,0,...,2954.326531,NaN,NaN,NaN,2952.942768,NaN,NaN,NaN,2951.794694,NaN
3,20210107_1301,2021-01-07,1301,2990.0,2995.0,2975.0,2989.0,20400,1.0,0,...,2957.628766,NaN,NaN,NaN,2955.269041,NaN,NaN,NaN,2953.253726,NaN
4,20210108_1301,2021-01-08,1301,2988.0,3020.0,2971.0,2999.0,29500,1.0,0,...,2961.568883,NaN,NaN,NaN,2958.090393,NaN,NaN,NaN,2955.047697,NaN
5,20210112_1301,2021-01-12,1301,3100.0,3130.0,3070.0,3130.0,64500,1.0,0,...,2977.609942,NaN,NaN,NaN,2969.181336,NaN,NaN,NaN,2961.908572,NaN
6,20210113_1301,2021-01-13,1301,3135.0,3140.0,3095.0,3110.0,28500,1.0,0,...,2990.218519,NaN,NaN,NaN,2978.266411,NaN,NaN,NaN,2967.716079,NaN
7,20210114_1301,2021-01-14,1301,3125.0,3125.0,3055.0,3065.0,30200,1.0,0,...,2997.340565,NaN,NaN,NaN,2983.862126,NaN,NaN,NaN,2971.531135,NaN
8,20210115_1301,2021-01-15,1301,3050.0,3060.0,3020.0,3025.0,21700,1.0,0,...,2999.974797,NaN,NaN,NaN,2986.516183,NaN,NaN,NaN,2973.627953,NaN
9,20210118_1301,2021-01-18,1301,3010.0,3080.0,3010.0,3050.0,16300,1.0,0,...,3004.739102,NaN,NaN,NaN,2990.611913,NaN,NaN,NaN,2976.622935,NaN


In [6]:
price_features.to_csv('validation_price_features.csv', index=False)